# Stint Validation and Lineup Summary


This notebook validates the reconstructed stint dataset and summarizes lineup behavior before applying performance metrics.

## Objective

With the stint-construction pipeline producing consistent five-player lineups, this notebook focuses on validating the reconstructed dataset and exploring its key properties.

The goal is to ensure that lineup transitions, scoring patterns, and substitution behavior align with expectations before using the dataset for player and lineup evaluation.

In [16]:
import pandas as pd
import sqlite3

pd.set_option("display.max_columns", None)

# Load stints
stints_df = pd.read_pickle("../data/interim/stints_prototype_game.pkl")

# Load substitution anomalies
sub_anomalies_df = pd.read_pickle("../data/interim/sub_anomalies_prototype_game.pkl")

## Lineup integrity validation

We verify that all reconstructed stints contain valid five-player lineups.

In [17]:
stints_df["home_n_players"] = stints_df["home_lineup"].apply(len)
stints_df["away_n_players"] = stints_df["away_lineup"].apply(len)

stints_df[["home_n_players", "away_n_players"]].value_counts()

home_n_players  away_n_players
5               5                 32
dtype: int64

## Stint distribution

We examine how the game is segmented into lineup stints.

In [23]:
print("Total number of reconstructed stints:", len(stints_df))
display(stints_df["period"].value_counts().sort_index())

Total number of reconstructed stints: 32


1    9
2    9
3    5
4    9
Name: period, dtype: int64

## Stint distribution interpretation

The game is segmented into 32 distinct lineup stints, reflecting changes in on-court personnel and game flow.

Stint counts are relatively balanced across most periods, with 9 stints observed in the 1st, 2nd, and 4th quarters. The 3rd quarter shows fewer stints (5), suggesting fewer lineup changes or longer periods of lineup stability during that segment.

This distribution is consistent with expected basketball dynamics, where substitutions and game interruptions create natural segmentation of play into lineup intervals.

## Stint duration analysis

To better understand how the game is segmented, we estimate the duration of each stint using the difference between the start and end game clocks within each period.

Because the game clock counts down, stint duration is calculated as:

- start time remaining in the period
- minus end time remaining in the period

This provides an approximate measure of how long each lineup segment remained on the court.

In [24]:
def clock_to_seconds(clock_str):
    if pd.isna(clock_str):
        return np.nan
    
    minutes, seconds = str(clock_str).split(":")
    return int(minutes) * 60 + int(seconds)

stints_df["start_clock_seconds"] = stints_df["start_clock"].apply(clock_to_seconds)
stints_df["end_clock_seconds"] = stints_df["end_clock"].apply(clock_to_seconds)

stints_df["stint_duration_seconds"] = (
    stints_df["start_clock_seconds"] - stints_df["end_clock_seconds"]
)

stints_df[[
    "period",
    "start_clock",
    "end_clock",
    "stint_duration_seconds"
]].head(10)

,period,start_clock,end_clock,stint_duration_seconds
0,1,12:00,7:35,265
1,1,7:33,4:35,178
2,1,4:21,3:57,24
3,1,3:57,3:57,0
4,1,3:57,3:09,48
5,1,3:09,2:13,56
6,1,1:55,1:28,27
7,1,1:25,0:21,64
8,1,0:21,0:00,21
9,2,11:45,10:32,73


In [25]:
stints_df["stint_duration_seconds"].describe()

count     32.000000
mean      82.593750
std       71.897265
min        0.000000
25%       33.000000
50%       59.500000
75%      123.500000
max      301.000000
Name: stint_duration_seconds, dtype: float64

## Duration interpretation

Stint durations vary depending on game flow, substitutions, and period boundaries.

Short stints typically occur around substitutions or end-of-period transitions, while longer stints indicate periods of lineup stability.

Overall, the distribution of stint durations appears consistent with expected basketball patterns, supporting the validity of the reconstructed lineup segments.

## Scoring behavior across stints

We analyze how scoring is distributed across lineup segments to verify that point accumulation aligns with the reconstructed stint boundaries.

In [26]:
stints_df[["home_points", "away_points"]].describe()

,home_points,away_points
count,32.000000,32.000000
mean,2.437500,2.562500
std,3.320877,2.850439
min,0.000000,0.000000
25%,0.000000,0.000000
50%,1.000000,2.000000
75%,3.000000,4.250000
max,14.000000,12.000000


In [27]:
stints_df[[
    "period",
    "home_points",
    "away_points",
    "home_lineup_names",
    "away_lineup_names"
]].head(10)

,period,home_points,away_points,home_lineup_names,away_lineup_names
0,1,10,6,"[Bo McCalebb, Linas Kleiza, Luka Zoric, Bojan Bogdanovic, Emir Preldzic]","[Thabo Sefolosha, Kevin Durant, Serge Ibaka, Reggie Jackson, Kendrick Perkins]"
1,1,10,6,"[Bo McCalebb, Linas Kleiza, Gasper Vidmar, Bojan Bogdanovic, Emir Preldzic]","[Thabo Sefolosha, Kevin Durant, Serge Ibaka, Reggie Jackson, Kendrick Perkins]"
2,1,0,3,"[Bo McCalebb, Linas Kleiza, Gasper Vidmar, Bojan Bogdanovic, Emir Preldzic]","[Thabo Sefolosha, Kevin Durant, Serge Ibaka, Reggie Jackson, Nick Collison]"
3,1,1,0,"[Bo McCalebb, Linas Kleiza, Gasper Vidmar, Bojan Bogdanovic, Emir Preldzic]","[Thabo Sefolosha, Serge Ibaka, Reggie Jackson, Jeremy Lamb, Nick Collison]"
4,1,0,0,"[Bo McCalebb, Melih Mahmutoglu, Linas Kleiza, Gasper Vidmar, Bojan Bogdanovic]","[Thabo Sefolosha, Serge Ibaka, Reggie Jackson, Jeremy Lamb, Nick Collison]"
5,1,3,0,"[Melih Mahmutoglu, Linas Kleiza, Gasper Vidmar, Kenan Sipahi, Bojan Bogdanovic]","[Thabo Sefolosha, Serge Ibaka, Reggie Jackson, Jeremy Lamb, Nick Collison]"
6,1,0,1,"[Melih Mahmutoglu, Linas Kleiza, Gasper Vidmar, Kenan Sipahi, Bojan Bogdanovic]","[Thabo Sefolosha, Hasheem Thabeet, Reggie Jackson, Jeremy Lamb, Nick Collison]"
7,1,1,1,"[Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vidmar, Kenan Sipahi, Bojan Bogdanovic]","[Thabo Sefolosha, Hasheem Thabeet, Reggie Jackson, Jeremy Lamb, Nick Collison]"
8,1,2,0,"[Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vidmar, James Birsen, Kenan Sipahi]","[Thabo Sefolosha, Hasheem Thabeet, Reggie Jackson, Jeremy Lamb, Nick Collison]"
9,2,1,2,"[Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vidmar, James Birsen, Kenan Sipahi]","[Thabo Sefolosha, Hasheem Thabeet, Reggie Jackson, Jeremy Lamb, Nick Collison]"


## Scoring interpretation

Scoring across stints reflects the natural variability of basketball possessions. Some stints contain no scoring, while others capture short bursts of offensive production.

The distribution of points across lineup segments aligns with expectations from play-by-play data, indicating that scoring has been correctly attributed to each stint.

This confirms that both stint boundaries and score tracking are functioning as intended.

## Substitution anomaly analysis

We review substitution events that required tolerant handling to understand how often inconsistencies occur and whether they impact the reconstructed lineup state.

In [28]:
sub_anomalies_df["action_taken"].value_counts()

applied_clean    29
no_change         3
added_only        2
removed_only      1
Name: action_taken, dtype: int64

## Substitution anomaly interpretation

The majority of substitutions are applied cleanly, indicating strong alignment between the play-by-play event stream and the reconstructed on-court lineup state.

A small number of substitutions require tolerant handling (e.g., player already present in the lineup or missing from the current state). These discrepancies are expected when working with real-world event data, particularly in cases of incomplete or noisy substitution reporting.

Importantly, these anomalies do not compromise the structural integrity of the lineup reconstruction. All such events are explicitly logged, providing transparency and enabling targeted refinement of the substitution logic in future iterations.

##  Lineup frequency analysis

We analyze how often each lineup appears throughout the game to identify the most frequently used player combinations.

This provides an initial view of lineup stability and rotation patterns, which are key components of lineup-based performance analysis.

In [29]:
home_lineup_counts = (
    stints_df["home_lineup_names"]
    .apply(tuple)
    .value_counts()
    .reset_index()
)

home_lineup_counts.columns = ["lineup", "stint_count"]

home_lineup_counts.head(10)

,lineup,stint_count
0,"(Bo McCalebb, Linas Kleiza, Gasper Vidmar, Bojan Bogdanovic, Emir Preldzic)",7
1,"(Bo McCalebb, Linas Kleiza, Luka Zoric, Bojan Bogdanovic, Emir Preldzic)",2
2,"(Melih Mahmutoglu, Linas Kleiza, Gasper Vidmar, Kenan Sipahi, Bojan Bogdanovic)",2
3,"(Melih Mahmutoglu, Izzet Turkyilmaz, James Birsen, Kenan Sipahi, Ayberk Olmaz)",2
4,"(Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vidmar, James Birsen, Kenan Sipahi)",2
5,"(Melih Mahmutoglu, Luka Zoric, James Birsen, Kenan Sipahi, Emir Preldzic)",2
6,"(Melih Mahmutoglu, Linas Kleiza, Luka Zoric, James Birsen, Kenan Sipahi)",2
7,"(Bo McCalebb, Baris Ermis, Luka Zoric, Bojan Bogdanovic, Emir Preldzic)",2
8,"(Baris Ermis, Luka Zoric, James Birsen, Kenan Sipahi, Bojan Bogdanovic)",2
9,"(Baris Ermis, Linas Kleiza, Luka Zoric, James Birsen, Kenan Sipahi)",1


In [30]:
away_lineup_counts = (
    stints_df["away_lineup_names"]
    .apply(tuple)
    .value_counts()
    .reset_index()
)

away_lineup_counts.columns = ["lineup", "stint_count"]

away_lineup_counts.head(10)

,lineup,stint_count
0,"(Thabo Sefolosha, Hasheem Thabeet, Reggie Jackson, Jeremy Lamb, Nick Collison)",6
1,"(Thabo Sefolosha, Serge Ibaka, Reggie Jackson, Jeremy Lamb, Kendrick Perkins)",6
2,"(Thabo Sefolosha, Jeremy Lamb, Perry Jones III, Kendrick Perkins, Derek Fisher)",5
3,"(Thabo Sefolosha, Serge Ibaka, Reggie Jackson, Jeremy Lamb, Nick Collison)",3
4,"(Thabo Sefolosha, Hasheem Thabeet, Jeremy Lamb, Perry Jones III, Diante Garrett)",3
5,"(Thabo Sefolosha, Kevin Durant, Serge Ibaka, Reggie Jackson, Kendrick Perkins)",2
6,"(Thabo Sefolosha, Serge Ibaka, Hasheem Thabeet, Reggie Jackson, Jeremy Lamb)",2
7,"(Thabo Sefolosha, Serge Ibaka, Jeremy Lamb, Perry Jones III, Kendrick Perkins)",2
8,"(Thabo Sefolosha, Serge Ibaka, Jeremy Lamb, Perry Jones III, Diante Garrett)",2
9,"(Thabo Sefolosha, Kevin Durant, Serge Ibaka, Reggie Jackson, Nick Collison)",1


## Lineup frequency interpretation

The distribution of lineup frequency shows how teams rotate players throughout the game.

Lineups with higher stint counts indicate more stable or preferred player combinations, while less frequent lineups reflect short-term adjustments, substitutions, or situational rotations.

This analysis provides an initial understanding of lineup usage patterns, which can be further explored in terms of performance and impact in later stages of the project.

In [31]:
stints_df["home_lineup_tuple"] = stints_df["home_lineup_names"].apply(tuple)
stints_df["away_lineup_tuple"] = stints_df["away_lineup_names"].apply(tuple)

all_lineups = pd.concat([
    stints_df["home_lineup_tuple"],
    stints_df["away_lineup_tuple"]
])

all_lineups.value_counts().head(10)

(Bo McCalebb, Linas Kleiza, Gasper Vidmar, Bojan Bogdanovic, Emir Preldzic)         7
(Thabo Sefolosha, Serge Ibaka, Reggie Jackson, Jeremy Lamb, Kendrick Perkins)       6
(Thabo Sefolosha, Hasheem Thabeet, Reggie Jackson, Jeremy Lamb, Nick Collison)      6
(Thabo Sefolosha, Jeremy Lamb, Perry Jones III, Kendrick Perkins, Derek Fisher)     5
(Thabo Sefolosha, Hasheem Thabeet, Jeremy Lamb, Perry Jones III, Diante Garrett)    3
(Thabo Sefolosha, Serge Ibaka, Reggie Jackson, Jeremy Lamb, Nick Collison)          3
(Melih Mahmutoglu, Linas Kleiza, Luka Zoric, James Birsen, Kenan Sipahi)            2
(Thabo Sefolosha, Serge Ibaka, Jeremy Lamb, Perry Jones III, Diante Garrett)        2
(Thabo Sefolosha, Serge Ibaka, Jeremy Lamb, Perry Jones III, Kendrick Perkins)      2
(Thabo Sefolosha, Serge Ibaka, Hasheem Thabeet, Reggie Jackson, Jeremy Lamb)        2
dtype: int64

## Quick interpretation of lineup frequency

**What we observe:**

- A small number of lineups appear multiple times (6–7 stints)
- Other lineups appear only 1–3 times

**This suggests:**

- Teams rely on **core lineups** that remain on the court for longer or across multiple segments of the game
- Additional lineups reflect **rotational or situational adjustments**, typically appearing for shorter periods

This pattern is consistent with real basketball dynamics, where coaches maintain stable core units while rotating players based on matchups, fatigue, and game context.

## Save lineup frequency summaries

To support downstream analysis and reproducibility, lineup frequency summaries are saved as intermediate artifacts.

These tables capture how often each lineup appears in the game and can be used in later notebooks for performance evaluation, visualization, and modeling.

Saving these outputs ensures that lineup usage patterns can be reused without recomputing them from the raw stint dataset.

In [33]:
home_lineup_counts.to_csv("../data/interim/home_lineup_counts.csv", index=False)
away_lineup_counts.to_csv("../data/interim/away_lineup_counts.csv", index=False)

## Key insights

The stint validation process confirms that the reconstructed dataset behaves consistently with expected basketball dynamics.

Lineup integrity is preserved across all stints, and scoring patterns align with the underlying play-by-play data, indicating that event ordering and score tracking are correctly implemented.

Lineup frequency analysis reveals that a small number of player combinations are used repeatedly throughout the game, reflecting core lineup units. Less frequent lineups correspond to substitutions, rotations, and situational adjustments.

Substitution anomaly analysis shows that most events are applied cleanly, with only a small number requiring tolerant handling. These cases are expected in real-world event data and do not compromise the structural validity of the dataset.

Overall, the dataset provides a reliable and interpretable representation of on-court lineup states, suitable for downstream performance analysis.

## Next step

With a validated stint dataset and a clear understanding of lineup behavior, the next stage of the project is to compute player and lineup performance metrics.

This includes:

- evaluating scoring performance at the lineup level
- estimating player on/off impact
- constructing basic efficiency metrics (e.g., point differential per stint)
- preparing features for more advanced impact modeling

These analyses will build directly on the reconstructed lineup states developed in this notebook, transforming the dataset from a structural representation into a performance evaluation framework.